## EDC Sentinel Hub - XCUBE integration

This notebook demonstrates the deep integration between Sentinel Hub and the xcube 

1. On-the-fly access to any SentinelHub datasets
2. Utilising band / flag mask information
3. Utilising external vector / feature data
4. Persisting data cube's in user's workspace
5. Using Xcube Viewer to visualize generated cubes

In [ ]:
# xcube_sh imports
from xcube_sh.cube import open_cube
from xcube_sh.config import CubeConfig
from xcube_sh.observers import Observers
from xcube_sh.viewer import ViewerServer

# xcube imports
from xcube.core.maskset import MaskSet
from xcube.core.geom import mask_dataset_by_geometry
from xcube.core.geom import clip_dataset_by_geometry

# Various utilities
import json
import numpy as np
import xarray as xr
import shapely.geometry
import IPython.display
import zarr

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

---
For this demo, we are focussing on small coastal area near Kiel in Northern Germany (Baltic Sea)

In [ ]:
x1 = 10.00  # degree
y1 = 54.27  # degree
x2 = 11.00  # degree
y2 = 54.60  # degree

bbox = x1, y1, x2, y2

In [ ]:
#IPython.display.GeoJSON(shapely.geometry.box(*bbox).__geo_interface__)

Later in this NB we are going to compute some indexes from bands atmospherically corrected bands B04, B05, B06, B11 of Sentinel-2 (S2L2A)
Our time range covers two and a half month of last year's summer: 2018-05-14 to 2018-07-31

The desired resolution is roughly 20 meters per pixel:

In [ ]:
spatial_res = 0.00018   # = 20.038 meters in degree>

---
Example (!): Sentinel-2 L2A with aggregated observations that fall into equal-size `time_period` of 2 days:

In [ ]:
cube_config = CubeConfig(dataset_name='S2L2A',
                         band_names=['B04', 'B05', 'B06', 'B11', 'SCL', 'CLD'],
                         tile_size=[512, 512],
                         bbox=bbox,
                         spatial_res=spatial_res,
                         time_range=['2018-05-14', '2018-07-31'],
                         time_period='2D')

We define a `request_collector` as an observer for SH requests made, so we can show SH usage stats. This is a developer tool, useful for demonstration purposes too. **Otherwise, this is not needed.**

In [ ]:
request_collector = Observers.request_collector()

Open a data cube:

In [ ]:
cube = open_cube(cube_config, observer=request_collector)

In [ ]:
cube

Note: a data cube equivalent to `cube` can be generated persistently using the following CLI command: 

```bash
$ xcsh cubify S2L1C -o cube.zarr -b B08 -g 10.237174,53.506205,10.271174,53.540205 -r 6.640625e-05 -t 2017-08-01,2017-08-31 -p 1D
$ python 
>>> import xarray as xr
>>> cube = xr.open_zarr('cube.zarr')
>>> cube
```

In [ ]:
cube.time

No requests have been made yet. Requests are made only if data is actually required.

In [ ]:
request_collector.stats

Band metadata also comprises wavelength info:

In [ ]:
cube.B04

In [ ]:
# SH requests
cube.B04.sel(time='2018-05-21 10:00:00', method='nearest').plot.imshow(vmin=0, vmax=0.2, cmap='Greys_r', figsize=(16, 10))

Now SentinelHub data requests have been made

In [ ]:
request_collector.stats

Scene classification flags have CF-conformant flag encoding information, which xcube can decode

In [ ]:
cube.SCL

In [ ]:
import xcube.core.maskset as maskset

scene_classif = maskset.MaskSet(cube.SCL)
print(scene_classif)

xcube mask sets also follow data cube structure:

In [ ]:
scene_classif.water

In [ ]:
# SH requests
scene_classif.water.sel(time='2018-05-21').plot.imshow(figsize=(16, 10))

In [ ]:
request_collector.stats

We can use any of the mask or combinations of it to mask entire cubes. Here we create a "water cube":

In [ ]:
water_cube = cube.where(scene_classif.water)
water_cube

In [ ]:
# SH requests
water_cube.B04.sel(time='2018-05-21').plot.imshow(vmin=0, vmax=0.05, cmap='Greys_r', figsize=(16, 10))

In [ ]:
request_collector.stats

---

We now compute a Chlorophyll indicator called *Maximum Chlorophyll Index* from bands B04, B05, B06:

In [ ]:
b_from = water_cube.B04
b_peek = water_cube.B05
b_to = water_cube.B06

wlen_from = b_from.attrs['wavelength']
wlen_peek = b_peek.attrs['wavelength']
wlen_to = b_to.attrs['wavelength']

f = (wlen_peek - wlen_from) / (wlen_to - wlen_from)
mci = (b_peek - b_from) - f * (b_to - b_from)

mci.attrs['long_name'] = 'Maximum Chlorophyll Index'
mci.attrs['units'] = 'unitless'
mci

---

And the well-known *Normalised Difference Vegetation Index*:

In [ ]:
veg_cube = cube.where(scene_classif.vegetation)
veg_cube

In [ ]:
b_red = veg_cube.B04
b_nir = veg_cube.B11
ndvi = (b_nir - b_red) / (b_nir + b_red)
ndvi.attrs['long_name'] = 'Normalized Difference Vegetation Index'
ndvi.attrs['units'] = 'unitless'
ndvi

In [ ]:
my_cube = xr.Dataset({'mci': mci, 'ndvi': ndvi})
my_cube

In [ ]:
# Plot mci index
my_cube.mci.isel(time=8).plot.imshow(vmin=0, vmax=0.005, cmap='viridis', figsize=(16, 10))

In [ ]:
request_collector.stats